# Bài 4: Deep Reinforcement Learning — Huấn luyện Agent bằng PPO

**Dựa theo:** Kaggle Learn — *Intro to Game AI and Reinforcement Learning*, bài "Deep Reinforcement Learning"
(gốc: https://www.kaggle.com/code/alexisbcook/deep-reinforcement-learning)

**Nối tiếp:** `01_Play_the_Game_ConnectX_VN.ipynb`, `02_One_Step_Lookahead_ConnectX_VN.ipynb`,
`03_N_Step_Lookahead_Minimax_ConnectX_VN.ipynb`

**Lưu ý phiên bản:** Notebook gốc của Kaggle dùng thư viện `stable-baselines` (TensorFlow 1.x, đã cũ và khó
cài trên môi trường hiện đại). Notebook này dùng **Stable-Baselines3** (PyTorch, đang được duy trì tích cực)
— thuật toán và ý tưởng hoàn toàn tương đương, chỉ khác cú pháp thư viện. Đây cũng là thư viện phổ biến để
bạn tham khảo cách triển khai PPO cho đồ án.

---

## Mục tiêu bài học

Đây là bài **quan trọng nhất** trong chuỗi 4 bài, vì nó kết nối trực tiếp tới phần lõi thuật toán trong đồ
án của bạn. Sau bài này, bạn sẽ:

1. Hiểu vì sao **heuristic viết tay** (bài 2, 3) không mở rộng được (scale) khi bài toán phức tạp hơn.
2. Hiểu ý tưởng cốt lõi của **Deep RL**: dùng mạng neural network để **tự học** hàm chính sách (policy) và
   hàm giá trị (value function), thay vì con người thiết kế thủ công.
3. Hiểu tổng quan thuật toán **PPO (Proximal Policy Optimization)** — thuật toán chính bạn dùng trong thesis.
4. Biết cách xây dựng một **custom Gym environment** bọc quanh `kaggle_environments` để tương thích với các
   thư viện RL chuẩn (Stable-Baselines3) — kỹ năng bạn sẽ áp dụng lại khi wrap môi trường mô phỏng giao
   thông (SUMO/Unity) thành Gym environment cho PPO/MAPPO.
5. Thiết kế **reward shaping** — một bước cực kỳ quan trọng, ảnh hưởng trực tiếp tới chất lượng agent học được.
6. Huấn luyện một PPO agent thực sự để chơi ConnectX, và so sánh với các agent rule-based ở bài 1–3.

> 💡 **Đây chính là bài học "bắc cầu" sang đồ án của bạn.** Toàn bộ quy trình ở đây — định nghĩa
> observation space, action space, reward function, wrap môi trường thành Gym interface, gọi PPO từ
> Stable-Baselines3 để train — là **chính xác quy trình bạn sẽ lặp lại** khi huấn luyện PPO/MAPPO cho các
> giao lộ trong đồ án, chỉ khác ở môi trường mô phỏng (SUMO/Unity Digital Twin thay vì kaggle_environments)
> và không gian trạng thái (dữ liệu từ YOLOv8 + ByteTrack thay vì bàn cờ).

## Phần 1 — Lý thuyết

### 1.1. Vì sao cần Deep RL? Hạn chế của heuristic viết tay

Ở bài 2–3, ta tự thiết kế hàm `get_heuristic()` bằng cách đếm số cửa sổ 2/3/4 quân liên tiếp, rồi gán trọng
số thủ công (`1e6, 1e2, 1, -1e2, -1e6`). Cách này có 2 hạn chế lớn:

1. **Không mở rộng được (không scale):** với ConnectX, ta còn "đoán" được công thức heuristic hợp lý. Nhưng
   với bài toán phức tạp hơn nhiều (ví dụ: trạng thái giao lộ với hàng chục xe, nhiều làn, nhiều pha đèn),
   không ai có thể viết tay một công thức heuristic chính xác và tối ưu.
2. **Heuristic con người thiết kế thường không tối ưu:** dù ta cố gắng, trọng số `1e6, 1e2, 1, -1e2, -1e6`
   vẫn chỉ là phỏng đoán hợp lý — không có gì đảm bảo đó là lựa chọn tốt nhất.

**Deep RL** giải quyết cả 2 vấn đề: thay vì viết tay hàm đánh giá, ta để một **mạng neural network** tự học
hàm đó (và cả chính sách hành động) thông qua **hàng nghìn ván tự chơi (self-play)**, tối ưu hoá trực tiếp
theo tín hiệu thắng/thua thực tế — không cần con người đoán công thức.

### 1.2. Các thành phần của bài toán RL (nhắc lại và mở rộng)

| Khái niệm | Trong ConnectX | Ký hiệu toán học |
|---|---|---|
| Trạng thái (state) | Bàn cờ hiện tại | `s` |
| Hành động (action) | Chọn cột để đánh | `a` |
| Chính sách (policy) | Xác suất chọn từng cột, do neural network tính | `π(a\|s)` |
| Hàm giá trị (value function) | Ước lượng "từ trạng thái này, kỳ vọng thắng/thua ra sao" | `V(s)` |
| Phần thưởng (reward) | +1 thắng, -1 thua, 0 hoà (hoặc tinh chỉnh — xem mục 1.4) | `r` |
| Lợi thế (advantage) | Hành động này tốt hơn/kém hơn trung bình bao nhiêu | `A(s,a) = Q(s,a) - V(s)` |

### 1.3. Actor-Critic và thuật toán PPO

**PPO (Proximal Policy Optimization)**, do OpenAI công bố năm 2017, là thuật toán **Actor-Critic**:

- **Actor** (chính là **policy network** `π(a|s)`): mạng neural network nhận trạng thái, trả về **phân phối
  xác suất** trên các hành động. Actor được cập nhật để tăng xác suất chọn các hành động có `advantage` dương.
- **Critic** (chính là **value network** `V(s)`): mạng neural network ước lượng "trạng thái này tốt cỡ nào",
  dùng để tính `advantage` cho Actor học, và để giảm phương sai (variance) khi ước lượng gradient.

**Điểm đặc trưng nhất của PPO** là **clipped surrogate objective** — cơ chế giới hạn mức độ thay đổi của
policy sau mỗi lần cập nhật:

```
L^CLIP(θ) = E[ min( r_t(θ) · A_t,  clip(r_t(θ), 1-ε, 1+ε) · A_t ) ]
```

trong đó `r_t(θ) = π_θ(a_t|s_t) / π_θ_old(a_t|s_t)` là tỷ lệ giữa policy mới và policy cũ. Việc "clip" (cắt)
tỷ lệ này trong khoảng `[1-ε, 1+ε]` (thường `ε=0.2`) ngăn không cho policy thay đổi quá đột ngột chỉ sau 1
lần cập nhật — giúp quá trình huấn luyện **ổn định hơn** so với các thuật toán policy gradient cũ hơn.

> 🔑 **Đây chính là thuật toán bạn dùng cho Tier 1 của đồ án.** Với **MAPPO** (Multi-Agent PPO), ý tưởng cốt
> lõi vẫn giữ nguyên (clipped surrogate objective), nhưng mỗi agent (mỗi đèn giao thông) có Actor riêng,
> trong khi Critic có thể **chia sẻ thông tin toàn cục** (centralized critic) để phối hợp tốt hơn giữa các
> giao lộ — đây là điểm khác biệt chính giữa PPO đơn-agent (bài học này) và MAPPO đa-agent (đồ án của bạn).

### 1.4. Reward shaping — thiết kế phần thưởng

Reward mặc định của ConnectX khá "thưa" (sparse): chỉ có giá trị khác 0 khi ván đấu **kết thúc** (thắng/thua/
hoà), còn suốt quá trình chơi thì reward = 0. Điều này khiến agent khó học vì không có tín hiệu phản hồi
tức thời cho từng nước đi.

**Reward shaping** là kỹ thuật thiết kế lại hàm reward để cung cấp tín hiệu học tốt hơn, ví dụ: thưởng nhỏ
khi tạo được 3 quân liên tiếp, phạt nhỏ khi để đối thủ tạo được 3 quân liên tiếp — dùng lại ý tưởng
`get_heuristic()` ở bài 2, nhưng lần này **không dùng nó để chọn nước đi trực tiếp**, mà dùng làm **tín hiệu
reward** cho agent tự học.

> ⚠️ **Cẩn trọng:** reward shaping sai có thể khiến agent học ra hành vi "lách luật" để tối đa hoá reward phụ
> thay vì mục tiêu thật (thắng ván). Đây là vấn đề **reward hacking** — cực kỳ quan trọng phải lưu ý khi bạn
> thiết kế reward cho PPO/MAPPO điều khiển đèn giao thông (ví dụ: agent có thể học cách "né" một số trạng
> thái khó thay vì thực sự tối ưu hoá lưu lượng, nếu reward thiết kế không cẩn thận).

## Phần 2 — Thực hành

### 2.1. Cài đặt thư viện

Ta cần: `kaggle_environments`, `gymnasium` (chuẩn Gym hiện đại), `stable-baselines3` (thư viện RL dựa trên
PyTorch, hỗ trợ PPO).

In [ ]:
!pip install kaggle_environments -q
!pip install stable-baselines3 -q
!pip install gymnasium -q


In [ ]:
from kaggle_environments import make, evaluate
import numpy as np
import random
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

env_kaggle = make("connectx", debug=True)
config = env_kaggle.configuration
print("Cấu hình môi trường:", config)


### 2.2. Xây dựng Custom Gym Environment (`ConnectFourGym`)

Đây là bước **quan trọng nhất về mặt kỹ thuật** trong bài học: `kaggle_environments` không tuân theo chuẩn
Gym mà Stable-Baselines3 yêu cầu, nên ta cần viết một lớp "bọc" (wrapper class) chuyển đổi qua lại.

Một Gym environment chuẩn cần định nghĩa:

- `observation_space`: mô tả **hình dạng và kiểu dữ liệu** của trạng thái agent quan sát được.
- `action_space`: mô tả **tập hành động** hợp lệ.
- `reset()`: đưa môi trường về trạng thái ban đầu, trả về observation đầu tiên.
- `step(action)`: thực hiện 1 hành động, trả về `(observation_mới, reward, terminated, truncated, info)`.

> 🔑 Đây chính xác là "hợp đồng giao diện" (interface contract) bạn sẽ phải cài đặt lại khi wrap môi trường
> mô phỏng giao thông (SUMO traci API / Unity ML-Agents) thành 1 Gym environment để MAPPO có thể huấn luyện.

In [ ]:
class ConnectFourGym(gym.Env):
    """
    Bọc môi trường kaggle_environments 'connectx' thành 1 Gym Environment chuẩn,
    để Stable-Baselines3 có thể huấn luyện PPO trực tiếp.
    """
    def __init__(self, agent_doi_thu="random"):
        super(ConnectFourGym, self).__init__()
        self.env = make("connectx", debug=True)
        # trainer cho phép agent của ta (đi quân đầu tiên) đấu với agent_doi_thu do kaggle_environments cung cấp
        self.trainer = self.env.train([None, agent_doi_thu])
        self.config = self.env.configuration
        self.rows = self.config.rows
        self.columns = self.config.columns

        # Action space: chọn 1 trong `columns` cột
        self.action_space = spaces.Discrete(self.columns)

        # Observation space: ảnh 1 kênh, kích thước (rows, columns), giá trị trong {0, 1, 2}
        self.observation_space = spaces.Box(
            low=0, high=2, shape=(1, self.rows, self.columns), dtype=np.int32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.trainer.reset()
        board = np.array(obs["board"]).reshape(1, self.rows, self.columns)
        return board, {}

    def step(self, action):
        # kaggle_environments không tự kiểm tra action hợp lệ trước khi truyền vào -> ta xử lý nước đi không hợp lệ ở đây
        is_valid = self.obs_board_top_free(action)

        if is_valid:
            obs, kaggle_reward, done, info = self.trainer.step(int(action))
            reward = self.shape_reward(kaggle_reward, done, obs)
        else:
            # Phạt nặng nếu chọn cột đã đầy, kết thúc episode sớm để agent học tránh hành vi này
            reward, done, info = -10, True, {}
            obs = self._last_obs

        self._last_obs = obs
        board = np.array(obs["board"]).reshape(1, self.rows, self.columns)
        terminated = done
        truncated = False
        return board, reward, terminated, truncated, info

    def obs_board_top_free(self, action):
        """Kiểm tra cột `action` còn ô trống ở hàng trên cùng không."""
        board = self._last_obs["board"] if hasattr(self, "_last_obs") else self.trainer.reset()["board"]
        return board[int(action)] == 0

    def shape_reward(self, kaggle_reward, done, obs):
        """
        Reward shaping: dựa trên reward gốc của kaggle_environments (1=thắng, -1=thua, 0=hoà/chưa kết thúc)
        + thưởng/phạt nhỏ theo heuristic để cung cấp tín hiệu học dày hơn (xem mục 1.4).
        """
        if done:
            if kaggle_reward == 1:
                return 1        # thắng
            elif kaggle_reward == -1:
                return -1       # thua
            else:
                return 1/2      # hoà - coi là kết quả trung tính, thiên nhẹ về tích cực để khuyến khích không thua
        else:
            return 1/100        # phần thưởng nhỏ cho mỗi bước còn sống (khuyến khích ván tiếp diễn thay vì random noise)


print("Đã định nghĩa xong lớp ConnectFourGym.")


### 2.3. Kiểm tra môi trường với `check_env`

Stable-Baselines3 cung cấp hàm `check_env` để kiểm tra Gym environment của bạn có tuân thủ đúng chuẩn không
(shape đúng, kiểu dữ liệu đúng, reset/step trả về đúng định dạng...). **Luôn chạy bước này trước khi train**
— tiết kiệm rất nhiều thời gian debug về sau, đặc biệt quan trọng khi bạn tự viết Gym wrapper cho môi trường
giao thông phức tạp hơn nhiều.

In [ ]:
test_env = ConnectFourGym(agent_doi_thu="random")

# check_env sẽ in cảnh báo/lỗi nếu environment không tuân thủ chuẩn Gym
try:
    check_env(test_env, warn=True)
    print("Environment hợp lệ theo chuẩn Gym!")
except Exception as e:
    print("Có vấn đề với environment:", e)


### 2.4. Khởi tạo và huấn luyện PPO agent

Ta dùng `CnnPolicy` — chính sách dạng Convolutional Neural Network, phù hợp với observation dạng "ảnh"
(bàn cờ ở đây được coi như 1 ảnh 1 kênh kích thước `rows x columns`). Đây cũng là kiểu policy network hợp lý
nếu trạng thái giao lộ của bạn được biểu diễn dạng lưới không gian (spatial grid).

⚠️ **Lưu ý về thời gian huấn luyện:** để PPO thực sự học tốt ConnectX cần hàng trăm nghìn đến hàng triệu
bước huấn luyện (`total_timesteps`), có thể mất từ vài phút đến vài giờ tuỳ phần cứng. Notebook này dùng số
bước nhỏ (`total_timesteps=10000`) chỉ để **minh hoạ quy trình chạy được**, không đủ để agent chơi giỏi.
Khi áp dụng thật cho đồ án, bạn cần huấn luyện lâu hơn nhiều và theo dõi bằng TensorBoard.

In [ ]:
# Tạo environment huấn luyện: agent của ta đấu với agent random
train_env = ConnectFourGym(agent_doi_thu="random")

# Khởi tạo model PPO với CnnPolicy
model = PPO(
    "CnnPolicy",
    train_env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=256,          # số bước thu thập trước mỗi lần cập nhật policy
    batch_size=64,
    n_epochs=4,           # số epoch tối ưu hoá trên mỗi batch dữ liệu thu thập được
    gamma=0.99,           # hệ số chiết khấu (discount factor) cho reward tương lai
    clip_range=0.2,       # chính là ε trong công thức clipped surrogate objective ở Phần 1
)

# Huấn luyện (số bước nhỏ để chạy nhanh minh hoạ - tăng lên khi áp dụng thật)
model.learn(total_timesteps=10000)

print("Huấn luyện xong!")


### 2.5. Đóng gói PPO model thành 1 agent tương thích `kaggle_environments`

Model PPO sau khi train trả về hành động qua `model.predict(obs)`, khác định dạng với agent function
`(obs, config) -> action` mà `kaggle_environments` yêu cầu. Ta viết 1 hàm chuyển đổi.

In [ ]:
def agent_ppo(obs, config):
    board = np.array(obs["board"]).reshape(1, config.rows, config.columns)
    action, _states = model.predict(board, deterministic=True)

    # Đảm bảo action hợp lệ (đề phòng agent PPO chọn nhầm cột đã đầy do chưa học đủ)
    valid_moves = [c for c in range(config.columns) if obs["board"][c] == 0]
    if int(action) not in valid_moves:
        return random.choice(valid_moves)
    return int(action)


print("Đã đóng gói xong agent_ppo, sẵn sàng thi đấu qua kaggle_environments!")


### 2.6. Đánh giá agent PPO so với các agent ở bài 1–3

In [ ]:
def agent_random(obs, config):
    valid_moves = [col for col in range(config.columns) if obs.board[col] == 0]
    return random.choice(valid_moves)


def get_win_percentages(agent1, agent2, n_rounds=20):
    cfg = {'rows': 6, 'columns': 7, 'inarow': 4}
    outcomes = evaluate("connectx", [agent1, agent2], cfg, [], n_rounds // 2)
    outcomes += [[b, a] for [a, b] in evaluate("connectx", [agent2, agent1], cfg, [], n_rounds - n_rounds // 2)]

    win_1 = np.round(outcomes.count([1, -1]) / len(outcomes) * 100, 1)
    win_2 = np.round(outcomes.count([-1, 1]) / len(outcomes) * 100, 1)
    draw = np.round(outcomes.count([0, 0]) / len(outcomes) * 100, 1)
    print(f"Agent 1 thắng: {win_1}%  |  Agent 2 thắng: {win_2}%  |  Hoà: {draw}%")


print("PPO agent (train 10k bước - minh hoạ) vs random:")
get_win_percentages(agent_ppo, agent_random, n_rounds=20)


> 🎯 **Kỳ vọng thực tế:** với chỉ 10.000 bước huấn luyện, agent PPO ở đây **chưa chắc** đã thắng nổi
> `agent_random` một cách áp đảo — vì PPO cần rất nhiều dữ liệu tương tác để hội tụ. Đây là điều hoàn toàn
> bình thường và là lý do quan trọng vì sao trong thực tế (kể cả cho đồ án), bạn cần: (1) huấn luyện đủ lâu
> (hàng trăm nghìn – hàng triệu bước), (2) theo dõi learning curve qua TensorBoard, (3) cân nhắc
> **warm-start bằng Behavior Cloning** (đúng như bạn đã lên kế hoạch cho đồ án) để rút ngắn thời gian hội tụ
> thay vì để PPO học từ đầu hoàn toàn ngẫu nhiên.

In [ ]:
# Xem 1 ván đấu cụ thể của agent PPO
env_kaggle.run([agent_ppo, agent_random])
env_kaggle.render(mode="ipython", width=500, height=450)
# Nếu không hiển thị được HTML, dùng: print(env_kaggle.render(mode="ansi"))


### 2.7. Theo dõi quá trình huấn luyện bằng TensorBoard (khuyến nghị)

Khi huấn luyện lâu hơn (hàng trăm nghìn bước trở lên), bạn **nên** bật TensorBoard để theo dõi các đường cong
quan trọng: `episode reward`, `value loss`, `policy loss`, `entropy`... Đây là thói quen bạn nên áp dụng
ngay từ đầu khi huấn luyện PPO/MAPPO cho đồ án — giúp phát hiện sớm các vấn đề như policy collapse, reward
không tăng, hoặc entropy giảm quá nhanh (mất khả năng khám phá).

In [ ]:
# Cách bật TensorBoard khi khởi tạo model:
#
# model = PPO(
#     "CnnPolicy",
#     train_env,
#     verbose=1,
#     tensorboard_log="./ppo_connectx_tensorboard/"
# )
# model.learn(total_timesteps=200_000, tb_log_name="ppo_run_1")
#
# Sau đó chạy trong terminal:
#   tensorboard --logdir ./ppo_connectx_tensorboard/
# rồi mở trình duyệt tới địa chỉ TensorBoard hiển thị (thường là http://localhost:6006)

print("Xem hướng dẫn TensorBoard trong comment phía trên.")


## Phần 3 — Bài tập thực hành

### Bài tập 1: Huấn luyện lâu hơn và so sánh

Tăng `total_timesteps` lên (ví dụ `100_000` nếu máy cho phép), huấn luyện lại, rồi so sánh tỷ lệ thắng với
phiên bản 10.000 bước ở trên. Bạn có thấy agent cải thiện rõ rệt không?

### Bài tập 2: Thử nghiệm reward shaping khác

Trong `shape_reward()`, thử các phương án khác, ví dụ:

- Bỏ hẳn phần thưởng nhỏ `1/100` mỗi bước — chỉ giữ reward thắng/thua/hoà (reward thưa/sparse thuần tuý).
- Thêm phần thưởng dựa trên `get_heuristic()` (từ bài 2) — ví dụ `reward += 0.001 * get_heuristic(...)`
  mỗi bước — để cung cấp tín hiệu học dày đặc hơn.

So sánh tốc độ hội tụ (agent học nhanh/chậm ra sao) giữa các phương án.

### Bài tập 3: Đấu với đối thủ khó hơn khi huấn luyện

Thử đổi `agent_doi_thu` khi khởi tạo `ConnectFourGym` từ `"random"` sang agent one-step lookahead (bài 2)
hoặc n-step lookahead (bài 3). Đây chính là ý tưởng **curriculum learning** — huấn luyện với đối thủ khó dần
để agent học chính sách mạnh hơn, thay vì chỉ học cách thắng agent random (dễ dẫn tới chính sách "lười",
không tổng quát hoá tốt).

In [ ]:
# TODO (Bài tập 3): dùng agent one-step lookahead ở bài 2 làm đối thủ khi huấn luyện
#
# def agent_one_step_lookahead(obs, config):
#     ... (copy từ bài 2)
#
# train_env_hard = ConnectFourGym(agent_doi_thu=agent_one_step_lookahead)
# model_v2 = PPO("CnnPolicy", train_env_hard, verbose=1)
# model_v2.learn(total_timesteps=50_000)


## Phần 4 — Liên hệ trực tiếp với đồ án PPO/MAPPO điều khiển đèn giao thông

Đây là bảng ánh xạ **chi tiết nhất** trong 4 bài, vì bài học này gần như là phiên bản thu nhỏ của pipeline
Tier 1 trong đồ án:

| Trong `ConnectFourGym` (ConnectX) | Trong đồ án điều khiển đèn giao thông |
|---|---|
| `self.env = make("connectx")` (kaggle_environments) | Môi trường mô phỏng: SUMO (qua traci API) hoặc Unity Digital Twin |
| `observation_space = Box(shape=(1, rows, columns))` — bàn cờ dạng ảnh | State space: ảnh/lưới mật độ xe theo làn, hoặc vector đặc trưng (queue length, waiting time, phase hiện tại...) trích xuất từ YOLOv8 + ByteTrack |
| `action_space = Discrete(columns)` — chọn 1 trong N cột | Action space: chọn pha đèn tiếp theo (Discrete) — có thể mở rộng thành MultiDiscrete cho nhiều giao lộ (MAPPO) |
| `step()` gọi `self.trainer.step(action)` | `step()` gọi `traci.simulationStep()` (SUMO) hoặc API tương ứng của Unity, rồi đọc trạng thái mới |
| `shape_reward()` — reward thắng/thua + tín hiệu nhỏ mỗi bước | Reward function: kết hợp giảm hàng đợi, giảm thời gian chờ trung bình, tăng throughput, phạt chuyển pha quá thường xuyên |
| `CnnPolicy` của Stable-Baselines3 | Policy network tự thiết kế (CNN nếu state dạng lưới không gian, hoặc MLP/Transformer nếu state dạng vector đặc trưng) |
| `PPO(...)` — 1 agent duy nhất | `MAPPO`: nhiều Actor (1 cho mỗi giao lộ) + Critic tập trung (centralized critic) dùng thông tin toàn cục để phối hợp |
| Huấn luyện từ đầu, policy ngẫu nhiên ban đầu | **Behavior Cloning warm-start**: khởi tạo policy ban đầu bằng cách bắt chước 1 chính sách rule-based/actuated control tốt, giúp PPO hội tụ nhanh hơn nhiều so với học từ đầu hoàn toàn ngẫu nhiên |
| `check_env()` kiểm tra Gym interface | Bước kiểm tra tương tự khi wrap SUMO/Unity — cực kỳ nên làm trước khi train hàng giờ, tránh phát hiện lỗi định dạng quá muộn |
| `total_timesteps`, TensorBoard log | Theo dõi tương tự cho MAPPO: reward theo thời gian, entropy, KL divergence giữa các lần cập nhật policy |

> ✅ **Gợi ý lộ trình tiếp theo cho đồ án của bạn:**
> 1. Trước khi động vào SUMO/Unity, hãy thử áp dụng **đúng quy trình notebook này** (wrap Gym, PPO đơn giản,
>    reward shaping cơ bản) trên 1 bài toán nhỏ hơn nếu có thể — để chắc chắn code chạy đúng interface.
> 2. Khi wrap môi trường giao thông thật, viết `check_env()` ngay từ đầu để bắt lỗi shape/kiểu dữ liệu sớm.
> 3. Bắt đầu với PPO đơn-agent cho 1 giao lộ trước khi mở rộng lên MAPPO nhiều giao lộ — dễ debug reward
>    shaping và hyperparameter hơn nhiều so với debug trực tiếp trên hệ multi-agent.
> 4. Luôn so sánh PPO/MAPPO đã train với **baseline rule-based** (fixed-time, hoặc agent kiểu one-step/n-step
>    lookahead như bài 2–3) để biết chắc RL agent thực sự học được điều gì đó có ích, không chỉ "chạy được".

---

## Tóm tắt toàn bộ chuỗi 4 bài học

| Bài | Cách tiếp cận | Ưu điểm | Nhược điểm |
|---|---|---|---|
| 1. Play the Game | Rule-based thủ công | Đơn giản, dễ hiểu | Chỉ xử lý được vài tình huống cụ thể |
| 2. One-Step Lookahead | Heuristic + nhìn 1 nước | Tổng quát hơn, phát hiện thắng/chặn tự động | Không tính phản ứng của đối thủ |
| 3. N-Step Lookahead | Minimax + heuristic | Nhìn xa hơn, chơi tốt hơn hẳn | Bùng nổ tổ hợp, chỉ hợp với game 2-người, state nhỏ |
| 4. Deep RL (PPO) | Neural network tự học policy/value | Mở rộng được tới bài toán phức tạp, không cần thiết kế heuristic hoàn chỉnh | Cần nhiều dữ liệu/thời gian huấn luyện, khó debug hơn, nhạy với reward shaping |

Chuỗi 4 bài này chính là hành trình thu nhỏ của lịch sử phát triển Game AI: từ luật cố định → tìm kiếm có
heuristic → tìm kiếm có đối kháng (Minimax) → và cuối cùng là Deep RL, nơi máy tự học thay vì con người viết
luật. Đồ án PPO/MAPPO của bạn đứng ở điểm cuối cùng của hành trình này, mở rộng thêm chiều **đa-agent hợp
tác** (nhiều đèn giao thông cùng phối hợp) — điều mà ConnectX (2 người chơi đối kháng) chưa từng đề cập tới.